In [1]:
# 1. Install specific version and ignore the dependency error
!pip install pyspark==3.5.0 -q

# 2. Install Java
!apt-get install openjdk-8-jdk-headless -qq > /dev/null

# 3. Set Environment Variables
import os
os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-8-openjdk-amd64"
os.environ["PYARROW_IGNORE_TIMEZONE"] = "1"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 316.9/316.9 MB 5.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 200.5/200.5 kB 13.5 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
dataproc-spark-connect 1.1.0 requires pyspark[connect]~=4.0.0, but you have pyspark 3.5.0 which is incompatible.


In [3]:
from pyspark.sql import SparkSession
from pyspark.sql.window import Window
from pyspark.sql.functions import (
    col, row_number, when, concat_ws, avg, max as spark_max,
    lit, coalesce, isnan, array, explode
)
from pyspark.sql.types import IntegerType

# We add a config to handle the timezone warning explicitly

spark = SparkSession.builder \
    .master("local[*]") \
    .appName("StevensGraduateCourse") \
    .getOrCreate()

print("Spark Session Created Successfully!")

Spark Session Created Successfully!


In [7]:
from google.colab import files

# Prompt to upload file
uploaded = files.upload()

Saving re_u.data to re_u.data


In [ ]:
# Example: Read the uploaded CSV into a Spark DataFrame
# Replace 'ratings.csv' with your actual filename
import io
df = spark.read.csv("ratings.csv", header=True, inferSchema=True)
df.show(5)

### Part 1

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql.types import IntegerType, FloatType
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.recommendation import ALS
# Initialize SparkSession
spark = SparkSession.builder.appName("RecommendationSystem").getOrCreate()

In [10]:
# Step 1: Load training data
training = spark.read.csv("trainItem.data", header=False)
training = training.withColumnRenamed("_c0", "userID") \
                   .withColumnRenamed("_c1", "itemID") \
                   .withColumnRenamed("_c2", "rating")

# Convert columns to appropriate data types
training = training.withColumn("userID", training["userID"].cast(IntegerType())) \
                   .withColumn("itemID", training["itemID"].cast(IntegerType())) \
                   .withColumn("rating", training["rating"].cast(FloatType()))

# Display the first few rows of training data
training.show(5)

+------+------+------+
|userID|itemID|rating|
+------+------+------+
|199808|248969|  90.0|
|199808|  2663|  90.0|
|199808| 28341|  90.0|
|199808| 42563|  90.0|
|199808| 59092|  90.0|
+------+------+------+
only showing top 5 rows



In [18]:
# Step 2: Configure ALS model
als = ALS(
    maxIter=5,
    rank=5,
    regParam=0.01,
    userCol="userID",
    itemCol="itemID",
    ratingCol="rating",
    nonnegative=True,
    implicitPrefs=False,
    coldStartStrategy="nan"
)

# Step 3: Train the ALS model
model = als.fit(training)

In [19]:
# Step 4: Load testing data
testing = spark.read.csv("testItem.data", header=False)
testing = testing.withColumnRenamed("_c0", "userID") \
                 .withColumnRenamed("_c1", "itemID") \
                 .withColumnRenamed("_c2", "rating")

# Convert columns to appropriate data types
testing = testing.withColumn("userID", testing["userID"].cast(IntegerType())) \
                 .withColumn("itemID", testing["itemID"].cast(IntegerType())) \
                 .withColumn("rating", testing["rating"].cast(FloatType()))

# Display the first few rows of testing data
testing.show(5)

+------+------+------+
|userID|itemID|rating|
+------+------+------+
|199810|208019|   0.0|
|199810| 74139|   0.0|
|199810|  9903|   0.0|
|199810|242681|   0.0|
|199810| 18515|   0.0|
+------+------+------+
only showing top 5 rows



In [20]:
# Step 5: Make predictions
predictions = model.transform(testing)

In [23]:
# Default score
default_rating = training.select(avg("rating").alias("avg_rating")).collect()[0]["avg_rating"]

# ALS score, with NaN fallback
als_scores = (
    testing
    .join(
        predictions.select("userID", "itemID", "prediction"),
        on=["userID", "itemID"],
        how="left"
    )
    .withColumn(
        "als_score",
        when(col("prediction").isNull() | isnan(col("prediction")), lit(float(default_rating)))
        .otherwise(col("prediction"))
    )
    .select("userID", "itemID", "als_score")
)

In [25]:
# Load track metadata
track_data = spark.read.csv("track_data.csv", header=True)

genre_cols = [f"Genre{i}" for i in range(1, 22)]

track_data = track_data.select(
    col("TrackID").cast(IntegerType()).alias("trackID"),
    col("AlbumID").cast(IntegerType()).alias("albumID"),
    col("ArtistID").cast(IntegerType()).alias("artistID"),
    *[col(c).cast(IntegerType()).alias(c) for c in genre_cols]
)

test_tracks = (
    testing
    .join(track_data, testing.itemID == track_data.trackID, how="left")
    .drop("trackID")
)

train_lookup = training.select(
    col("userID").alias("tr_userID"),
    col("itemID").alias("tr_itemID"),
    col("rating").alias("tr_rating")
)

# User's rating for candidate track's artist
artist_scores = (
    test_tracks
    .select("userID", "itemID", col("artistID").alias("metaID"))
    .join(
        train_lookup,
        (col("userID") == col("tr_userID")) & (col("metaID") == col("tr_itemID")),
        how="left"
    )
    .groupBy("userID", "itemID")
    .agg(avg("tr_rating").alias("artist_score"))
)

# User's rating for candidate track's album
album_scores = (
    test_tracks
    .select("userID", "itemID", col("albumID").alias("metaID"))
    .join(
        train_lookup,
        (col("userID") == col("tr_userID")) & (col("metaID") == col("tr_itemID")),
        how="left"
    )
    .groupBy("userID", "itemID")
    .agg(avg("tr_rating").alias("album_score"))
)

# User's rating for candidate track's genres
genre_long = (
    test_tracks
    .select("userID", "itemID", explode(array(*[col(c) for c in genre_cols])).alias("metaID"))
    .where(col("metaID").isNotNull())
)

genre_scores = (
    genre_long
    .join(
        train_lookup,
        (col("userID") == col("tr_userID")) & (col("metaID") == col("tr_itemID")),
        how="left"
    )
    .groupBy("userID", "itemID")
    .agg(
        avg("tr_rating").alias("genre_avg_score"),
        spark_max("tr_rating").alias("genre_max_score")
    )
)

# Global popularity of the track itself
track_global_scores = (
    training
    .groupBy("itemID")
    .agg(avg("rating").alias("track_global_score"))
)


In [40]:
# Hybrid score
scored = (
    als_scores
    .join(artist_scores, on=["userID", "itemID"], how="left")
    .join(album_scores, on=["userID", "itemID"], how="left")
    .join(genre_scores, on=["userID", "itemID"], how="left")
    .join(track_global_scores, on="itemID", how="left")
    .withColumn(
        "final_score",
        0.35 * coalesce(col("artist_score"), lit(float(default_rating))) +
        0.25 * coalesce(col("album_score"), lit(float(default_rating))) +
        0.20 * coalesce(col("genre_max_score"), lit(float(default_rating))) +
        0.10 * coalesce(col("track_global_score"), lit(float(default_rating))) +
        0.10 * col("als_score")
    )
)

In [41]:
# Rank 6 tracks per user
windowSpec = Window.partitionBy("userID").orderBy(col("final_score").desc(), col("itemID").asc())

submission = (
    scored
    .withColumn("rank", row_number().over(windowSpec))
    .withColumn("Predictor", when(col("rank") <= 3, 1).otherwise(0))
    .withColumn("TrackID", concat_ws("_", col("userID"), col("itemID")))
    .select("TrackID", "Predictor")
)

print("Submission row count:", submission.count())

submission.toPandas().to_csv("submission_hybrid_als.csv", index=False)
print("Saved submission_hybrid_als.csv")

Submission row count: 120000
Saved submission_hybrid_als.csv


### Part 2

In [38]:
from pyspark.sql import SparkSession
from pyspark.sql.types import IntegerType, FloatType
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col
from pyspark.sql.types import IntegerType, FloatType

# Reuse or create Spark session
spark = SparkSession.builder.appName("ALS_MSE_Experiments").getOrCreate()
sc = spark.sparkContext

# ----------------------------
# Load re_u.data
# Format: userID,itemID,rating
# ----------------------------
raw_data = spark.read.csv("re_u.data", header=False)

ratings = (
    raw_data
    .withColumnRenamed("_c0", "userID")
    .withColumnRenamed("_c1", "itemID")
    .withColumnRenamed("_c2", "rating")
    .withColumn("userID", col("userID").cast(IntegerType()))
    .withColumn("itemID", col("itemID").cast(IntegerType()))
    .withColumn("rating", col("rating").cast(FloatType()))
    .select("userID", "itemID", "rating")
)

ratings.show(5)
print("Total rows:", ratings.count())

evaluator = RegressionEvaluator(
    metricName="mse",
    labelCol="rating",
    predictionCol="prediction"
)

+------+------+------+
|userID|itemID|rating|
+------+------+------+
|   196|   242|   3.0|
|   186|   302|   3.0|
|    22|   377|   1.0|
|   244|    51|   2.0|
|   166|   346|   1.0|
+------+------+------+
only showing top 5 rows

Total rows: 100000


In [44]:
def train_and_eval(train_df, test_df, rank_value, iter_value):
    als = ALS(
        maxIter=iter_value,
        rank=rank_value,
        regParam=0.1,
        userCol="userID",
        itemCol="itemID",
        ratingCol="rating",
        coldStartStrategy="drop"
    )

    model = als.fit(train_df)
    predictions = model.transform(test_df)
    mse = evaluator.evaluate(predictions)

    return mse

In [45]:
# Scenario 1:
# Fix maxIter = 20
# Test rank = 5, 7, 10, 20
train_data, test_data = ratings.randomSplit([0.8, 0.2], seed=42)

rank_values = [5, 7, 10, 20]
scenario_1_results = []

print("\nScenario 1: fixed maxIter = 20, varying rank")

for r in rank_values:
    mse = train_and_eval(train_data, test_data, rank_value=r, iter_value=20)
    scenario_1_results.append((r, 20, mse))
    print(f"rank={r}, maxIter=20, MSE={mse}")


Scenario 1: fixed maxIter = 20, varying rank
rank=5, maxIter=20, MSE=0.8317668653666405
rank=7, maxIter=20, MSE=0.829195042636815
rank=10, maxIter=20, MSE=0.8409857153471296
rank=20, maxIter=20, MSE=0.8467186253052444


In [46]:
# Scenario 2:
# Fix rank = 20
# Test maxIter = 2, 5, 10, 20
iter_values = [2, 5, 10, 20]
scenario_2_results = []

print("\nScenario 2: fixed rank = 20, varying maxIter")

for it in iter_values:
    mse = train_and_eval(train_data, test_data, rank_value=20, iter_value=it)
    scenario_2_results.append((20, it, mse))
    print(f"rank=20, maxIter={it}, MSE={mse}")


Scenario 2: fixed rank = 20, varying maxIter
rank=20, maxIter=2, MSE=0.9139812866738292
rank=20, maxIter=5, MSE=0.8573099767159882
rank=20, maxIter=10, MSE=0.849028399192076
rank=20, maxIter=20, MSE=0.8467186253052411


In [47]:
# Scenario 3:
# Fix rank = 20 and maxIter = 20
# Test different data sizes
data_sizes = [2000, 5000, 10000, 20000, 50000, 100000]
scenario_3_results = []

print("\nScenario 3: fixed rank = 20, maxIter = 20, varying data size")

data_rdd = sc.textFile("re_u.data")

for size in data_sizes:
    pData = data_rdd.take(size)

    parsed = [
        tuple(line.split(","))
        for line in pData
    ]

    size_df = spark.createDataFrame(parsed, ["userID", "itemID", "rating"])

    size_df = (
        size_df
        .withColumn("userID", size_df["userID"].cast(IntegerType()))
        .withColumn("itemID", size_df["itemID"].cast(IntegerType()))
        .withColumn("rating", size_df["rating"].cast(FloatType()))
    )

    size_train, size_test = size_df.randomSplit([0.8, 0.2], seed=42)

    mse = train_and_eval(size_train, size_test, rank_value=20, iter_value=20)
    scenario_3_results.append((size, 20, 20, mse))

    print(f"data_size={size}, rank=20, maxIter=20, MSE={mse}")



Scenario 3: fixed rank = 20, maxIter = 20, varying data size
data_size=2000, rank=20, maxIter=20, MSE=2.71838507934871
data_size=5000, rank=20, maxIter=20, MSE=1.5838156851915886
data_size=10000, rank=20, maxIter=20, MSE=1.2531338173995803
data_size=20000, rank=20, maxIter=20, MSE=1.1056913190002462
data_size=50000, rank=20, maxIter=20, MSE=0.9542385199103706
data_size=100000, rank=20, maxIter=20, MSE=0.8441825967191011


In [48]:
# Summary tables
print("\nScenario 1 Results: rank vs MSE")
for r, it, mse in scenario_1_results:
    print(f"rank={r}, maxIter={it}, MSE={mse}")

print("\nScenario 2 Results: maxIter vs MSE")
for r, it, mse in scenario_2_results:
    print(f"rank={r}, maxIter={it}, MSE={mse}")

print("\nScenario 3 Results: data size vs MSE")
for size, r, it, mse in scenario_3_results:
    print(f"data_size={size}, rank={r}, maxIter={it}, MSE={mse}")


Scenario 1 Results: rank vs MSE
rank=5, maxIter=20, MSE=0.8317668653666405
rank=7, maxIter=20, MSE=0.829195042636815
rank=10, maxIter=20, MSE=0.8409857153471296
rank=20, maxIter=20, MSE=0.8467186253052444

Scenario 2 Results: maxIter vs MSE
rank=20, maxIter=2, MSE=0.9139812866738292
rank=20, maxIter=5, MSE=0.8573099767159882
rank=20, maxIter=10, MSE=0.849028399192076
rank=20, maxIter=20, MSE=0.8467186253052411

Scenario 3 Results: data size vs MSE
data_size=2000, rank=20, maxIter=20, MSE=2.71838507934871
data_size=5000, rank=20, maxIter=20, MSE=1.5838156851915886
data_size=10000, rank=20, maxIter=20, MSE=1.2531338173995803
data_size=20000, rank=20, maxIter=20, MSE=1.1056913190002462
data_size=50000, rank=20, maxIter=20, MSE=0.9542385199103706
data_size=100000, rank=20, maxIter=20, MSE=0.8441825967191011
